# FinSight – Exploratory Data Analysis

IEEE-CIS Fraud Detection Dataset deep-dive.
Download data from: https://www.kaggle.com/competitions/ieee-fraud-detection

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print('Libraries loaded')

In [ ]:
# Load data
txn = pd.read_csv('../data/raw/train_transaction.csv')
try:
    idn = pd.read_csv('../data/raw/train_identity.csv')
    df = txn.merge(idn, on='TransactionID', how='left')
    print(f'Merged shape: {df.shape}')
except FileNotFoundError:
    df = txn
    print(f'Transaction only shape: {df.shape}')

print(f'Fraud rate: {df["isFraud"].mean():.4f}')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center')

# Amount by class
df.groupby('isFraud')['TransactionAmt'].apply(lambda x: np.log1p(x)).hist(
    bins=50, alpha=0.6, ax=axes[1]
)
axes[1].set_title('Log(Amount+1) by Class')
axes[1].legend(['Legitimate', 'Fraud'])
plt.tight_layout()
plt.savefig('../reports/eda/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Missing value analysis
missing = df.isnull().mean().sort_values(ascending=False)
print('Top 20 missing columns:')
print(missing.head(20))

fig, ax = plt.subplots(figsize=(14, 6))
missing.head(30).plot.bar(ax=ax, color='coral')
ax.set_title('Missing Value Rate (Top 30 Features)')
ax.set_ylabel('Fraction Missing')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Transaction amount analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall distribution
df['TransactionAmt'].hist(bins=100, ax=axes[0,0], color='steelblue', alpha=0.7)
axes[0,0].set_title('Amount Distribution (raw)')
axes[0,0].set_xlabel('Amount ($)')

# Log distribution
np.log1p(df['TransactionAmt']).hist(bins=100, ax=axes[0,1], color='steelblue', alpha=0.7)
axes[0,1].set_title('Amount Distribution (log scale)')
axes[0,1].set_xlabel('log(1 + Amount)')

# By class
for label, grp in df.groupby('isFraud'):
    np.log1p(grp['TransactionAmt']).hist(
        bins=60, ax=axes[1,0], alpha=0.6,
        label='Fraud' if label else 'Legitimate'
    )
axes[1,0].set_title('Log Amount by Fraud Label')
axes[1,0].legend()

# Box plot
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[1,1])
axes[1,1].set_title('Amount Box Plot by Class')

plt.tight_layout()
plt.show()

print('Amount statistics by class:')
df.groupby('isFraud')['TransactionAmt'].describe()

In [ ]:
# Time analysis
from ml.feature_engineering import _engineer_time_features

df_time = _engineer_time_features(df.copy())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fraud rate by hour
fraud_by_hour = df_time.groupby('hour')['isFraud'].mean()
fraud_by_hour.plot.bar(ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Fraud Rate by Hour of Day')
axes[0,0].set_ylabel('Fraud Rate')

# Transaction volume by hour
df_time.groupby('hour').size().plot.bar(ax=axes[0,1], color='coral')
axes[0,1].set_title('Transaction Volume by Hour')

# Fraud rate by day of week
df_time.groupby('day_of_week')['isFraud'].mean().plot.bar(
    ax=axes[1,0], color='steelblue',
    tick_label=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
)
axes[1,0].set_title('Fraud Rate by Day of Week')

# Night vs day fraud
df_time.groupby('is_night')['isFraud'].mean().plot.bar(
    ax=axes[1,1], color=['steelblue', 'crimson'],
    tick_label=['Day', 'Night']
)
axes[1,1].set_title('Fraud Rate: Night vs Day')

plt.tight_layout()
plt.show()

In [ ]:
# Card type analysis
if 'card4' in df.columns and 'card6' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df.groupby('card4')['isFraud'].mean().sort_values(ascending=False).plot.bar(
        ax=axes[0], color='steelblue'
    )
    axes[0].set_title('Fraud Rate by Card Network')
    axes[0].set_ylabel('Fraud Rate')
    
    df.groupby('card6')['isFraud'].mean().sort_values(ascending=False).plot.bar(
        ax=axes[1], color='coral'
    )
    axes[1].set_title('Fraud Rate by Card Type')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Email domain analysis
if 'P_emaildomain' in df.columns:
    top_domains = df.groupby('P_emaildomain').agg(
        count=('isFraud', 'count'),
        fraud_rate=('isFraud', 'mean')
    ).sort_values('count', ascending=False).head(20)
    
    print('Top 20 email domains with fraud rates:')
    display(top_domains)

In [ ]:
# Full feature engineering and quick model validation
from ml.feature_engineering import load_and_engineer, time_based_split
from pathlib import Path

print('Running feature engineering pipeline...')
X, y, feature_cols = load_and_engineer(Path('../data/raw'))
print(f'Features: {len(feature_cols)}, Samples: {len(X)}')

X_train, X_val, y_train, y_val = time_based_split(X, y)
print(f'Train: {len(X_train)}, Val: {len(X_val)}')
print(f'Train fraud rate: {y_train.mean():.4f}')
print(f'Val fraud rate:   {y_val.mean():.4f}')